In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
!file "/content/drive/MyDrive/ASR/audio-RU.wav"

/content/drive/MyDrive/ASR/audio-RU.wav: RIFF (little-endian) data, WAVE audio, Microsoft PCM, 16 bit, stereo 44100 Hz


In [ ]:
import torch
# import librosa
# from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

In [31]:
MODEL = "openai/whisper-large-v3-turbo"
AUDIO_PATH = "/content/drive/MyDrive/ASR/audio-RU.wav"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 0.Предобработка

Whisper обучался исключительно на аудио частотой дискретизации 16 кГц. У PyAnnote 3.1 такие же требования к аудио

In [32]:
CHUNK_LEN = 30 # требования whisper
OVERLAP_LEN = 7
LANG = "ru"

In [33]:
audio, sr = librosa.load(AUDIO_PATH, sr=16000)
audio_len = len(audio) / sr

chunk_samples = CHUNK_LEN * sr
overlap_samples = OVERLAP_LEN * sr
step = chunk_samples - overlap_samples

In [34]:
chunk_samples

480000

In [35]:
step

400000

Так как длина аудио у нас больше 30 сек, разобьем его на равные чанки (без использования VAD)

In [36]:
chunks = []
starts = []

start = 0
while start < len(audio):
    end = min(start + chunk_samples, len(audio))
    chunks.append(audio[start:end])
    starts.append(start / sr)   # время в секундах
    start += step

# 1. Инференс Whisper

In [37]:
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
).to(DEVICE)

Распознаем каждый чанк

In [38]:
def process_chunk_and_extract_segments(chunk, sr, processor, model, device, lang, offset):
    segments = []
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=lang, task="transcribe") # whisper output

    inputs = processor(chunk, sampling_rate=sr, return_tensors="pt").to(DEVICE, torch.float16)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            forced_decoder_ids=forced_decoder_ids,
            return_timestamps=True
        )[0]

    tokens = processor.tokenizer.convert_ids_to_tokens(generated_ids)

    current_text = []
    start_time = None

    for tok in tokens:
        if tok.startswith("<|") and tok.endswith("|>") and tok[2:-2].replace('.', '').isdigit():
            try:
                ts = float(tok[2:-2]) + offset
            except ValueError:
                continue

            if current_text and start_time is not None:
                text = processor.tokenizer.convert_tokens_to_string(current_text).strip()
                if text:
                    segments.append({"start": start_time, "end": ts, "text": text})
                current_text = []
            start_time = ts

        elif tok not in processor.tokenizer.all_special_tokens:
            current_text.append(tok)

    if current_text and start_time is not None:
        text = processor.tokenizer.convert_tokens_to_string(current_text).strip()
        if text:
            est_end = start_time + max(0.2, len(current_text) * 0.02)
            segments.append({"start": start_time, "end": est_end, "text": text})

    return segments

In [39]:
all_segments = []

for chunk_index, chunk in enumerate(chunks):
    print(f"Обрабатываем чанк {chunk_index+1}/{len(chunks)}")
    offset = starts[chunk_index]

    chunk_segments = process_chunk_and_extract_segments(chunk, sr, processor, model, DEVICE, LANG, offset)
    all_segments.extend(chunk_segments)

Обрабатываем чанк 1/10
Обрабатываем чанк 2/10
Обрабатываем чанк 3/10
Обрабатываем чанк 4/10
Обрабатываем чанк 5/10
Обрабатываем чанк 6/10
Обрабатываем чанк 7/10
Обрабатываем чанк 8/10
Обрабатываем чанк 9/10
Обрабатываем чанк 10/10


Удаляем перекрытия

In [40]:
clean_segments = []
for seg in all_segments:
    if not clean_segments or seg["start"] >= clean_segments[-1]["end"] - 0.5:
        clean_segments.append(seg)
    else:
        clean_segments[-1]["end"] = max(clean_segments[-1]["end"], seg["end"])
        clean_segments[-1]["text"] += " " + seg["text"]

Создаем временные метки для вывода

In [42]:
def sec_to_hms(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

In [53]:
file_content = []
file_content.append("Транскрипция")

for seg in clean_segments:
    start_hms = sec_to_hms(seg["start"])
    end_hms = sec_to_hms(seg["end"])
    segment_line = f"[{start_hms}-{end_hms}] {seg['text']}"
    file_content.append(segment_line)

print(f"Успешно транскрибировано")

Успешно транскрибировано


In [54]:
output_filename = "/content/transcript_with_timestamps.txt"
with open(output_filename, "w", encoding="utf-8") as f:
    f.writelines(file_content)

In [57]:
file_content[:10]

['Транскрипция',
 '[00:00:00-00:00:02] поменяйте, и мы будущую версию тоже что-нибудь поменяем.',
 '[00:00:02-00:00:03] Поэтому мы просто эти',
 '[00:00:03-00:00:05] ворнинги подавляем, говорим, что',
 '[00:00:05-00:00:07] мы хотим их заигнорить. Ну, а здесь',
 '[00:00:07-00:00:09] просто выбираем стиль нашего будущего графика.',
 '[00:00:10-00:00:12] Дальше. Грузим',
 '[00:00:12-00:00:14] датасет. Это тот же самый',
 '[00:00:14-00:00:15] Титаник. Мы берем просто посылочек,',
 '[00:00:16-00:00:16] фигачим, и нормально.']